In [3]:
!pip install jiwer

  Using cached jiwer-4.0.0-py3-none-any.whl (23 kB)
  Using cached rapidfuzz-3.14.6-cp311-cp311-win_amd64.whl (1.7 MB)



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import json
import re
import unicodedata
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from jiwer import wer

In [8]:
# ============================================================
# Paths
# ============================================================

TRANSCRIPTION_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\demo\transcription\transcription_demo.json")
GROUNDTRUTH_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\results\uit_viquad_train_squad.json")
OUTPUT_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\demo\transcription\audio_context_mapping_phoasr_VIQUAD_demo.json")
REJECTED_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\demo\transcription\rejected_mapping_phoasr_VIQUAD_demo.json")
TOP_K = 5

Matches audio context files in speech_data with corresponding id field in transcription.json. Done by reading 2 json files and match the closest

In [9]:
# Maximum acceptable WER for a match.
#
# Example:
#   0.30 = 30% WER
#
# Increase this if too many legitimate matches are rejected.
# Decrease this if unrelated audio files are being matched.
MAX_WER = 2

# Number of candidates retrieved by TF-IDF before WER
# verification.
TOP_K = 5


# ============================================================
# Text normalization
# ============================================================

def normalize_text(text):
    """
    Normalize text before matching.

    Vietnamese diacritics are preserved.
    """

    text = unicodedata.normalize("NFC", text)
    text = text.lower()

    # Remove punctuation
    text = re.sub(
        r"[^\w\sÀ-ỹ]",
        " ",
        text,
        flags=re.UNICODE
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# Extract context ID prefix
# ============================================================

def get_context_id_prefix(qa_id):

    parts = qa_id.split("-")

    if len(parts) < 2:
        return None

    return f"{parts[0]}-{parts[1]}"

In [11]:

# ============================================================
# Load ASR transcriptions
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcriptions = json.load(f)


# ============================================================
# Load ViSQA JSON
# ============================================================

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth = json.load(f)


# ============================================================
# Extract contexts
# ============================================================

contexts = []

for data_item in groundtruth["data"]:

    for paragraph in data_item["paragraphs"]:

        context = paragraph["context"]
        qas = paragraph["qas"]

        if not context or not qas:
            continue

        # Get context ID from the first QA.
        #
        # Example:
        # 0057-0077-0001
        #       ↓
        # 0057-0077
        context_id = get_context_id_prefix(
            qas[0]["id"]
        )

        contexts.append({
            "context_id": context_id,
            "context": context,
            "qas": qas
        })


print(f"Loaded {len(contexts)} unique contexts")
print(f"Loaded {len(transcriptions)} audio transcriptions")


# ============================================================
# Normalize contexts
# ============================================================

normalized_contexts = [
    normalize_text(item["context"])
    for item in contexts
]


# ============================================================
# Character n-gram TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    analyzer="char",

    # 3-5 character n-grams
    ngram_range=(3, 5),

    min_df=1,

    # Helps reduce the effect of very frequent n-grams
    sublinear_tf=True
)

context_vectors = vectorizer.fit_transform(
    normalized_contexts
)


# ============================================================
# Storage for results
# ============================================================

results = []
rejected_files = []


# ============================================================
# Match every audio transcription
# ============================================================

for i, item in enumerate(transcriptions):

    audio_file = item["file"]
    transcription = item["transcription"]

    normalized_transcription = normalize_text(
        transcription
    )

    # --------------------------------------------------------
    # Handle empty transcription
    # --------------------------------------------------------

    if not normalized_transcription:

        rejected_entry = {
            "audio_file": audio_file,
            "transcription": transcription,
            "reason": "empty_transcription"
        }

        rejected_files.append(rejected_entry)

        continue


    # --------------------------------------------------------
    # Convert ASR transcription to TF-IDF vector
    # --------------------------------------------------------

    transcription_vector = vectorizer.transform(
        [normalized_transcription]
    )


    # --------------------------------------------------------
    # Calculate cosine similarity against all contexts
    # --------------------------------------------------------

    similarities = cosine_similarity(
        transcription_vector,
        context_vectors
    )[0]


    # --------------------------------------------------------
    # Retrieve top-k candidates
    # --------------------------------------------------------

    top_indices = similarities.argsort()[-TOP_K:][::-1]


    candidates = []


    # --------------------------------------------------------
    # Calculate WER for candidates
    # --------------------------------------------------------

    for idx in top_indices:

        candidate_context = normalized_contexts[idx]

        similarity = similarities[idx]

        candidate_wer = wer(
            candidate_context,
            normalized_transcription
        )

        candidates.append({
            "context_id": contexts[idx]["context_id"],
            "similarity": float(similarity),
            "wer": float(candidate_wer)
        })


    # --------------------------------------------------------
    # Select candidate with lowest WER
    # --------------------------------------------------------

    best_candidate = min(
        candidates,
        key=lambda x: x["wer"]
    )

    best_context_id = best_candidate["context_id"]

    best_context = contexts[
        next(
            idx
            for idx in top_indices
            if contexts[idx]["context_id"] == best_context_id
        )
    ]
    # --------------------------------------------------------
    # Apply WER cutoff
    # --------------------------------------------------------

    if best_candidate["wer"] > MAX_WER:

        rejected_entry = {
            "audio_file": audio_file,
            "transcription": transcription,
            "matched_context": best_context["context"],
            "best_context_id": best_context_id,

            "best_similarity": (
                best_candidate["similarity"]
            ),

            "best_wer": best_candidate["wer"],

            "wer_cutoff": MAX_WER,

            "reason": "best_candidate_exceeds_wer_cutoff",

            "candidates": candidates
        }

        rejected_files.append(rejected_entry)

    else:

        # ----------------------------------------------------
        # Find matched context
        # ----------------------------------------------------

        best_context = next(
            context
            for context in contexts
            if context["context_id"] == best_context_id
        )


        # ----------------------------------------------------
        # Store accepted result
        # ----------------------------------------------------

        results.append({
            "audio_file": audio_file,

            "transcription": transcription,

            "context_id": best_context_id,

            "matched_context": best_context["context"],

            "qas": best_context["qas"],

            "similarity": best_candidate["similarity"],

            "wer": best_candidate["wer"],

            "candidates": candidates
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 10 == 0:

        print(
            f"Processed "
            f"{i + 1}/{len(transcriptions)}"
        )


# ============================================================
# Save accepted matches
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Save rejected files
# ============================================================

with open(REJECTED_FILE, "w", encoding="utf-8") as f:

    json.dump(
        rejected_files,
        f,
        ensure_ascii=False,
        indent=2)

Loaded 4101 unique contexts
Loaded 28 audio transcriptions
Processed 10/28
Processed 20/28


In [12]:
# ============================================================
# Summary
# ============================================================

total = len(transcriptions)

accepted = len(results)

rejected = len(rejected_files)

acceptance_rate = (
    accepted / total * 100
    if total > 0
    else 0
)

rejection_rate = (
    rejected / total * 100
    if total > 0
    else 0
)


print()
print("=" * 60)
print("MATCHING COMPLETE")
print("=" * 60)

print(f"Total audio files:     {total}")
print(f"Accepted matches:      {accepted}")
print(f"Rejected files:        {rejected}")

print(
    f"Acceptance rate:       "
    f"{acceptance_rate:.2f}%"
)

print(
    f"Rejection rate:        "
    f"{rejection_rate:.2f}%"
)

print()
print(f"WER cutoff:             {MAX_WER:.2f}")
print()
print(f"Accepted output:        {OUTPUT_FILE}")
print(f"Rejected output:        {REJECTED_FILE}")
print("=" * 60)


MATCHING COMPLETE
Total audio files:     28
Accepted matches:      28
Rejected files:        0
Acceptance rate:       100.00%
Rejection rate:        0.00%

WER cutoff:             2.00

Accepted output:        G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\demo\transcription\audio_context_mapping_phoasr_VIQUAD_demo.json
Rejected output:        G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\demo\transcription\rejected_mapping_phoasr_VIQUAD_demo.json


Field replacement

In [7]:
import json
import pathlib as Path

file1 = r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\ft_vit5_visqa_test_results.json"
file2 = r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\wer_calc_VISQA_VIQUAD_val.json"
output_file = r"G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\ft_VISQA_VIQUAD_val.json"

# Load file 1
with open(file1, "r", encoding="utf-8") as f:
    data1 = json.load(f)

# Load file 2
with open(file2, "r", encoding="utf-8") as f:
    data2 = json.load(f)

# Replace transcription in file 2 with vit5_prediction from file 1
for i, result2 in enumerate(data2["results"]):
    result2["transcription"] = data1["results"][i]["vit5_prediction"]

# Save as a new file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data2, f, ensure_ascii=False, indent=2)

print(f"Saved updated file to: {output_file}")

Saved updated file to: G:\Uni\Sem2\NLP\term_paper\data\ViSQA\ViSQA\transcription\ft_VISQA_VIQUAD_val.json
